# Latent Diffusion 실습: 노이즈를 더하고 잠재 공간에서 생각하기

forward noise와 latent space의 계산량 절감 직관을 확인합니다.


In [ ]:
# numpy는 노이즈와 다운샘플링 계산, matplotlib은 이미지를 보여주는 데 사용합니다.
import numpy as np
import matplotlib.pyplot as plt

# 확산 과정은 난수를 쓰므로 시드를 고정합니다.
np.random.seed(44)


In [ ]:
# 원형 물체가 있는 32x32 장난감 이미지를 만듭니다.
h, w = 32, 32
yy, xx = np.mgrid[:h, :w]
center = np.array([16, 16])
radius = 8
image = (((yy - center[0]) ** 2 + (xx - center[1]) ** 2) < radius ** 2).astype(float)
plt.imshow(image, cmap="gray")
plt.title("clean image x0")
plt.axis("off")
plt.show()


In [ ]:
# x_t = sqrt(alpha_bar_t) * x0 + sqrt(1-alpha_bar_t) * noise 를 구현합니다.
def add_noise(x0, alpha_bar):
    # alpha_bar가 작아질수록 원본 비중은 줄고 노이즈 비중은 커집니다.
    noise = np.random.normal(0, 1, x0.shape)
    xt = np.sqrt(alpha_bar) * x0 + np.sqrt(1 - alpha_bar) * noise
    return xt, noise

alpha_bars = [0.95, 0.7, 0.35, 0.05]
noisy_images = [add_noise(image, a)[0] for a in alpha_bars]
fig, axes = plt.subplots(1, len(noisy_images), figsize=(12, 3))
for ax, xt, alpha_bar in zip(axes, noisy_images, alpha_bars):
    ax.imshow(xt, cmap="gray")
    ax.set_title(f"alpha_bar={alpha_bar}")
    ax.axis("off")
plt.show()


In [ ]:
# Latent Diffusion은 픽셀 공간보다 작은 공간에서 확산을 수행합니다.
def average_pool2d(x, factor=4):
    # factor x factor 영역 평균으로 이미지를 압축합니다.
    h, w = x.shape
    return x.reshape(h // factor, factor, w // factor, factor).mean(axis=(1, 3))

def nearest_upsample(x, factor=4):
    # 작은 latent를 다시 큰 이미지처럼 보이게 단순 반복합니다.
    return np.repeat(np.repeat(x, factor, axis=0), factor, axis=1)

latent = average_pool2d(image, factor=4)
latent_noisy, _ = add_noise(latent, alpha_bar=0.35)
decoded_preview = nearest_upsample(latent_noisy, factor=4)
print("원본 픽셀 수:", image.size)
print("잠재 표현 원소 수:", latent.size)
print("계산량 절감 비율:", image.size / latent.size, "배")
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, data, title in zip(axes, [image, latent, decoded_preview], ["pixel space", "latent space", "noisy latent preview"]):
    ax.imshow(data, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.show()


## 관찰 포인트

확산 모델은 원본을 점점 망가뜨리는 과정을 만들고, 학습된 모델은 그 노이즈를 제거하는 방향을 배웁니다.
